In [ ]:
"""Setup: load from data/processed/raw"""

from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

CSV_FILE = (Path.cwd() / "../data/raw/TruckInspection_Anonymous_sample_xlsx_-_Sheet1.csv.csv").resolve()

print(f"CSV_FILE : {CSV_FILE}")
print(f"Exists   : {CSV_FILE.exists()}")

# ── Qualtrics exports have two header rows:
#    Row 0  →  human-readable question labels   (becomes column names)
#    Row 1  →  Qualtrics import-ID metadata      (skip)
#    Row 2+ →  actual respondent data
raw = pd.read_csv(CSV_FILE, skiprows=[1])   # keep row 0 as column names, drop row 1
df  = raw.iloc[1:].reset_index(drop=True)   # drop the remaining metadata row

print(f"\nRows × Columns: {df.shape}")

CSV_FILE : /Users/johncao/Documents/GitHub/Inspection-Research/data/raw/TruckInspection_Anonymous_sample_xlsx_-_Sheet1.csv
Exists   : False


FileNotFoundError: [Errno 2] No such file or directory: '/Users/johncao/Documents/GitHub/Inspection-Research/data/raw/TruckInspection_Anonymous_sample_xlsx_-_Sheet1.csv'

In [ ]:
# Fetch Question Labels

for i, col in enumerate(df.columns):
    print(f"{i:3d}  {col}")

NameError: name 'df' is not defined

In [ ]:
# Quick Look on first 5 cols

KEY_COLS = [
    "StartDate", "EndDate", "Duration (in seconds)",
    "Q5",    # consent
    "Q7",    # email
    "Q8",    # organization
    "Q9",    # job title
    "Q11",   # inspection licenses
    "Q37",   # major / field of study
    "Q2",    # factors used in decision-making
    "score",
    "total_score1",
]

df[KEY_COLS].head()

NameError: name 'df' is not defined

In [4]:
# Random sample of 5 respondents

df[KEY_COLS].sample(5, random_state=42)

NameError: name 'df' is not defined

In [5]:
# Row and column counts 

print(f"Respondents  : {len(df)}")
print(f"Total columns: {df.shape[1]}")

NameError: name 'df' is not defined

In [ ]:
# Respondent profiles

print("=== Q11: Inspection licenses ===")
print(df["Q11"].value_counts().to_string(), "\n")

print("=== Q37: Field of study ===")
print(df["Q37"].value_counts().to_string(), "\n")

df["duration_min"] = pd.to_numeric(df["Duration (in seconds)"], errors="coerce") / 60
print("=== Survey duration (minutes) ===")
print(df["duration_min"].describe().round(1))

=== Q11: Inspection licenses ===


NameError: name 'df' is not defined

In [7]:
# Q2

factor_counts = Counter()
for row in df["Q2"].dropna():
    for f in row.split(","):
        factor_counts[f.strip()] += 1

factor_df = (
    pd.DataFrame.from_dict(factor_counts, orient="index", columns=["count"])
    .sort_values("count", ascending=False)
)
factor_df["pct_respondents"] = (factor_df["count"] / len(df) * 100).round(1)

print(factor_df.to_string())

# ── bar chart
fig, ax = plt.subplots(figsize=(9, 4))
factor_df["count"].plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of respondents")
ax.set_title("Q2 — Factors used in inspection decision-making")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [8]:
# Block responses (B1–B5)
# BxRes_N stores the inspector's decision for scenario block x, truck N.
# Values are semicolon-delimited component tags, e.g. 'brake;tire;light;'.


N_BLOCKS = 5

for b in range(1, N_BLOCKS + 1):
    res_cols = [f"B{b}Res_{n}" for n in range(1, 9)]
    block_data = df[res_cols].stack().dropna()
    value_counts = block_data.value_counts()
    n_respondents = df[f"B{b}Res_1"].notna().sum()
    print(f"\n=== Block {b}  ({n_respondents} respondents) ===")
    print(value_counts.head(10).to_string())

NameError: name 'df' is not defined

In [9]:
# Pass / Fail breakdown per block

# A decision of 'no;' = pass (no defect found).
# Anything else = fail (defect flagged).


N_BLOCKS = 5
summary = []

for b in range(1, N_BLOCKS + 1):
    res_cols = [f"B{b}Res_{n}" for n in range(1, 9)]
    block_data = df[res_cols].stack().dropna()
    n_pass = (block_data == "no;").sum()
    n_fail = (block_data != "no;").sum()
    summary.append({"block": f"B{b}", "pass": n_pass, "fail": n_fail})

summary_df = pd.DataFrame(summary).set_index("block")
print(summary_df)

summary_df.plot(kind="bar", figsize=(8, 4), color=["#4caf50", "#f44336"])
plt.title("Pass vs Fail decisions per block")
plt.xlabel("Block")
plt.ylabel("Decision count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [10]:
# Accuracy scores per respondent (total_score1)

# total_score1 is a comma-separated list of per-trial scores (0–1).
# We parse each respondent's scores and compute their mean accuracy.


def parse_scores(s):
    """Return list of floats from a comma-separated score string."""
    if pd.isna(s):
        return []
    return [float(x) for x in str(s).split(",") if x.strip() != ""]

df["scores_list"]  = df["total_score1"].apply(parse_scores)
df["mean_accuracy"] = df["scores_list"].apply(lambda lst: sum(lst) / len(lst) if lst else None)

print("=== Mean accuracy per respondent ===")
print(df["mean_accuracy"].describe().round(3))

df["mean_accuracy"].dropna().plot(
    kind="hist", bins=10, figsize=(7, 4),
    color="steelblue", edgecolor="white"
)
plt.xlabel("Mean accuracy (0 = worst, 1 = best)")
plt.title("Distribution of respondent accuracy scores")
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [11]:
# Does field of study (Q37) relate to accuracy?"""

# Simplify multi-discipline entries to their first listed major
df["major_primary"] = df["Q37"].str.split(",").str[0].str.strip()

accuracy_by_major = (
    df.dropna(subset=["mean_accuracy"])
      .groupby("major_primary")["mean_accuracy"]
      .agg(["mean", "count"])
      .sort_values("mean", ascending=False)
      .round(3)
)
accuracy_by_major.columns = ["mean_accuracy", "n"]
print(accuracy_by_major.to_string())

accuracy_by_major["mean_accuracy"].plot(
    kind="barh", figsize=(8, 4), color="steelblue"
)
plt.xlabel("Mean accuracy")
plt.title("Accuracy by primary field of study")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined